# MNIST pipeline: look at the data, train a few epochs, plot the curves

Run the cells from top to bottom. The settings in the second cell choose the model, the number of epochs and the size of the data subset. The same notebook runs on this computer's CPU (small settings) and on a Colab machine (larger settings), so you can test the pipeline quickly first and then scale up.

In [ ]:
# @title Setup and settings
import json, os, sys, time
for path in ("..", "/content/mnist"):  # project root on this computer, uploaded folder on Colab
    if os.path.isfile(os.path.join(path, "models.py")) and os.path.abspath(path) not in sys.path:
        sys.path.insert(0, os.path.abspath(path))
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import models, quicktrain

ON_COLAB = os.path.isdir("/content")
DATA_DIR = "/content/data" if ON_COLAB else "../data"
FIG_DIR = "/content/figures" if ON_COLAB else "figures"
os.makedirs(FIG_DIR, exist_ok=True)

# ---- Settings you can change ----
MODEL = "cnn"                                                      # "cnn" or "mlp"
EPOCHS = int(os.environ.get("MNIST_EPOCHS", "3" if ON_COLAB else "2"))
TRAIN_SIZE = 20000 if ON_COLAB else 2000                           # images used for training
VAL_SIZE = 5000 if ON_COLAB else 1000                              # images held out to measure progress
BATCH_SIZE = 64
LR = 1e-3
SEED = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| torch", torch.__version__, "| model:", MODEL, "| epochs:", EPOCHS)

In [ ]:
# @title Load the images
normalise = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
raw = datasets.MNIST(DATA_DIR, train=True, download=True)                       # plain images, for display
full = datasets.MNIST(DATA_DIR, train=True, download=True, transform=normalise)  # normalised, for training
order = torch.randperm(len(full), generator=torch.Generator().manual_seed(SEED)).tolist()
train_idx, val_idx = order[:TRAIN_SIZE], order[TRAIN_SIZE:TRAIN_SIZE + VAL_SIZE]  # no image is in both sets
train_loader = DataLoader(Subset(full, train_idx), batch_size=BATCH_SIZE, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(Subset(full, val_idx), batch_size=500)
print("training images:", len(train_idx), "| validation images:", len(val_idx), "| whole set:", len(full))

In [ ]:
# @title Look at the images
fig, axes = plt.subplots(4, 8, figsize=(10, 5.4))
for ax, i in zip(axes.flat, train_idx[:32]):
    image, label = raw[i]
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label), fontsize=10)
    ax.axis("off")
fig.suptitle("The first 32 training images, with their labels")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "1_samples.png"), dpi=110)
plt.show()

counts = torch.bincount(torch.tensor([raw.targets[i].item() for i in train_idx]), minlength=10)
plt.figure(figsize=(6, 2.6))
plt.bar(range(10), counts.tolist(), color="#b56a00")
plt.xticks(range(10)); plt.xlabel("digit"); plt.ylabel("images"); plt.title("Images per digit in the training subset")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "2_class_counts.png"), dpi=110); plt.show()

In [ ]:
# @title Build the model and check one batch
torch.manual_seed(SEED)
model = models.build_model(MODEL).to(device)
x, y = next(iter(train_loader))
print("parameters:", sum(p.numel() for p in model.parameters()))
print("one batch: images", tuple(x.shape), "| labels", y[:10].tolist(), "...")
print("model output for 2 images:", tuple(model(x[:2].to(device)).shape))

In [ ]:
# @title Train a few epochs
started = time.time()
history = quicktrain.fit(model, train_loader, val_loader, EPOCHS, device, LR)
print(f"trained {EPOCHS} epochs in {time.time() - started:.1f} s")
print("epoch | train loss | train acc | val loss | val acc")
for k in range(EPOCHS):
    print(f"{k + 1:5d} | {history['train_loss'][k]:10.4f} | {history['train_acc'][k]:9.4f} | "
          f"{history['val_loss'][k]:8.4f} | {history['val_acc'][k]:7.4f}")

In [ ]:
# @title Plot the loss and accuracy curves
steps_per_epoch = len(train_loader)
window = max(1, len(history["step_loss"]) // 25)
smooth = [sum(history["step_loss"][max(0, i - window + 1):i + 1]) / len(history["step_loss"][max(0, i - window + 1):i + 1])
          for i in range(len(history["step_loss"]))]
epoch_steps = [steps_per_epoch * (k + 1) for k in range(EPOCHS)]
epochs = list(range(1, EPOCHS + 1))

fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 4))
ax_loss.plot(history["step_loss"], color="#cccccc", label="training loss, every batch")
ax_loss.plot(smooth, color="#b56a00", label="training loss, smoothed")
ax_loss.plot(epoch_steps, history["val_loss"], "o-", color="#132030", label="validation loss, each epoch")
ax_loss.set_xlabel("training step"); ax_loss.set_ylabel("loss"); ax_loss.set_title("Loss"); ax_loss.legend()
ax_acc.plot(epochs, history["train_acc"], "o-", color="#b56a00", label="training accuracy")
ax_acc.plot(epochs, history["val_acc"], "o-", color="#132030", label="validation accuracy")
ax_acc.set_xticks(epochs); ax_acc.set_xlabel("epoch"); ax_acc.set_ylabel("accuracy"); ax_acc.set_title("Accuracy"); ax_acc.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "3_curves.png"), dpi=110)
plt.show()

In [ ]:
# @title Look at the predictions (wrong ones in red)
model.eval()
xs, ys = next(iter(val_loader))
with torch.no_grad():
    guesses = model(xs.to(device)).argmax(1).cpu()
model.train()
wrong = int((guesses != ys).sum())
fig, axes = plt.subplots(3, 8, figsize=(10, 4.4))
for ax, image, truth, guess in zip(axes.flat, xs[:24], ys[:24], guesses[:24]):
    ax.imshow(image[0], cmap="gray")
    ax.set_title(f"{guess.item()} (is {truth.item()})", fontsize=9, color="green" if guess == truth else "red")
    ax.axis("off")
fig.suptitle(f"Predicted digit and true digit; {wrong} of the first {len(ys)} validation images are wrong")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "4_predictions.png"), dpi=110)
plt.show()

In [ ]:
# @title Save the history
out = os.path.join(FIG_DIR, "history.json")
with open(out, "w") as handle:
    json.dump({"model": MODEL, "epochs": EPOCHS, "train_size": TRAIN_SIZE, "val_size": VAL_SIZE, "lr": LR,
               "device": str(device), "history": history}, handle, indent=2)
print("saved", out, "and the four figures in", FIG_DIR)